# ACDC Testing Dataset - Segmentation and Evaluation

This notebook processes the ACDC testing dataset (50 patients) with cardiac segmentation models and computes Dice scores against ground truth.

In [ ]:
# Install required packages (skip if already installed or if network is unavailable)
# Note: Run this on a login node if compute nodes don't have internet access
# import sys
# import subprocess

# packages = ['imageio', 'matplotlib', 'numpy', 'SimpleITK', 'torch', 'monai', 'Pillow', 'tqdm', 'omegaconf', 'pandas']

# # Map package names to their import names (if different)
# import_names = {
#     'Pillow': 'PIL',
#     'SimpleITK': 'SimpleITK',
# }

# try:
#     for package in packages:
#         import_name = import_names.get(package, package.lower().replace('-', '_'))
#         try:
#             __import__(import_name)
#             print(f"✓ {package} already installed")
#         except ImportError:
#             print(f"Installing {package}...")
#             subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', package])
#             print(f"✓ {package} installed successfully")
# except Exception as e:
#     print(f"⚠ Warning: Some packages may not be installed. Error: {e}")
#     print("If on a compute node without internet, please install packages on a login node first.")

In [ ]:
"""ACDC dataset segmentation and evaluation."""

import json
import math
import os
import statistics
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import SimpleITK as sitk  # noqa: N813
import torch
from medpy.metric import binary
from monai.transforms import Compose, SpatialPadd
from tqdm import tqdm

# Import the official ACDC preprocessing function
from cinema.data.acdc.preprocess import preprocess_pid

In [ ]:
def dice_score(pred: np.ndarray, gt: np.ndarray, label: int) -> float:
    """Calculate Dice score for a specific label.
    
    Args:
        pred: predicted segmentation (x, y, z)
        gt: ground truth segmentation (x, y, z)
        label: class label to compute Dice for
        
    Returns:
        Dice score value
    """
    pred_mask = (pred == label).astype(np.float32)
    gt_mask = (gt == label).astype(np.float32)
    
    intersection = np.sum(pred_mask * gt_mask)
    union = np.sum(pred_mask) + np.sum(gt_mask)
    
    if union == 0:
        return 1.0 if intersection == 0 else 0.0
    
    return 2.0 * intersection / union


def hd95_score(pred: np.ndarray, gt: np.ndarray, label: int, voxelspacing=None) -> float:
    """Calculate 95th percentile Hausdorff distance for a specific label.
    
    Args:
        pred: predicted segmentation (x, y, z)
        gt: ground truth segmentation (x, y, z)
        label: class label to compute HD95 for
        voxelspacing: voxel spacing for physical distance calculation
        
    Returns:
        HD95 value in mm (or voxels if no spacing provided)
    """
    pred_mask = (pred == label).astype(np.uint8)
    gt_mask = (gt == label).astype(np.uint8)
    
    # Both masks are empty - perfect agreement
    if pred_mask.sum() == 0 and gt_mask.sum() == 0:
        return 0.0
    
    # One is empty, the other is not - return infinity
    if pred_mask.sum() == 0 or gt_mask.sum() == 0:
        return np.inf
    
    # Both masks have content - compute HD95
    return binary.hd95(pred_mask, gt_mask, voxelspacing=voxelspacing)

In [ ]:
# Debug tracking helpers (CASC-style report artifacts)
try:
    import resource  # Linux/Unix only
except ImportError:
    resource = None


def _safe_float(value):
    if value is None:
        return float("nan")
    try:
        value = float(value)
    except (TypeError, ValueError):
        return float("nan")
    return value if math.isfinite(value) else float("nan")


def _nan_to_na(value, precision=4):
    value = _safe_float(value)
    if math.isnan(value):
        return "NA"
    return f"{value:.{precision}f}"


def _get_peak_rss_gb():
    if resource is None:
        return float("nan")
    usage = resource.getrusage(resource.RUSAGE_SELF)
    # On Linux ru_maxrss is KiB.
    return _safe_float(usage.ru_maxrss) / (1024.0 * 1024.0)


class NotebookDebugTracker:
    def __init__(self):
        self.workflow_start_iso = datetime.now().astimezone().isoformat()
        self._workflow_t0 = time.perf_counter()
        self.records = []
        self.stage_t0 = {}

    def start_task(self, process_name, gpu_task=False):
        return {
            "process": process_name,
            "gpu_task": bool(gpu_task),
            "start_perf": time.perf_counter(),
            "start_peak_rss_gb": _get_peak_rss_gb(),
        }

    def end_task(self, task_handle, success=True):
        end_perf = time.perf_counter()
        runtime_s = _safe_float(end_perf - task_handle["start_perf"])
        peak_rss_gb = max(_safe_float(task_handle["start_peak_rss_gb"]), _safe_float(_get_peak_rss_gb()))
        self.records.append(
            {
                "process": task_handle["process"],
                "success": bool(success),
                "runtime_s": runtime_s,
                "cpu_percent": float("nan"),
                "peak_rss_gb": peak_rss_gb,
                "gpu_task": bool(task_handle["gpu_task"]),
            }
        )

    def workflow_duration_seconds(self):
        return _safe_float(time.perf_counter() - self._workflow_t0)


DEBUG_TRACKER = NotebookDebugTracker()

In [ ]:
def run_acdc_inference(
    preprocessed_dir: Path,
    patient_id: str,
    trained_dataset: str,
    seed: int,
    device: torch.device,
    dtype: torch.dtype,
    local_model_dir: Path,
    predictions_dir: Path,
    debug_tracker=None,
    ) -> dict:
    """Run segmentation inference on a preprocessed ACDC patient.
    
    Args:
        preprocessed_dir: directory containing preprocessed patient data
        patient_id: patient ID (e.g., 'patient101')
        trained_dataset: dataset model was trained on (acdc, mnms, mnms2)
        seed: random seed for the model
        device: torch device
        dtype: data type for inference
        local_model_dir: path to local model directory (Path object)
        predictions_dir: directory to save predictions
        debug_tracker: optional NotebookDebugTracker for CASC-style profiling
        
    Returns:
        dictionary with Dice scores and metadata
    """
    # Load model
    from safetensors.torch import safe_open
    from omegaconf import OmegaConf
    from cinema.segmentation.convunetr import get_model
    
    model_name = f"{trained_dataset}_seed{seed}"
    view = "sax"
    model_path = local_model_dir / f"finetuned/segmentation/{trained_dataset}_{view}/{trained_dataset}_{view}_{seed}.safetensors"
    config_path = local_model_dir / f"finetuned/segmentation/{trained_dataset}_{view}/config.yaml"
    
    load_task = None
    if debug_tracker is not None:
        load_task = debug_tracker.start_task(f"LOAD_MODEL ({model_name})", gpu_task=False)
    
    try:
        # Load state dict from safetensors
        state_dict = {}
        with safe_open(str(model_path), framework="pt", device="cpu") as f:
            for key in f.keys():
                state_dict[key] = f.get_tensor(key)
        
        # Load config and initialize model
        config = OmegaConf.load(str(config_path))
        model = get_model(config)
        model.load_state_dict(state_dict)
        model.eval()
        model.to(device)
        if load_task is not None:
            debug_tracker.end_task(load_task, success=True)
    except Exception:
        if load_task is not None:
            debug_tracker.end_task(load_task, success=False)
        raise
    
    # Load preprocessed 4D volume
    patient_dir = preprocessed_dir / patient_id
    image_path = patient_dir / f"{patient_id}_sax_t.nii.gz"
    ed_gt_path = patient_dir / f"{patient_id}_sax_ed_gt.nii.gz"
    es_gt_path = patient_dir / f"{patient_id}_sax_es_gt.nii.gz"
    
    # Read images (already preprocessed: resampled, cropped, normalized)
    image_sitk = sitk.ReadImage(str(image_path))
    images = np.transpose(sitk.GetArrayFromImage(image_sitk))  # (x, y, z, t)
    
    # Load ground truth
    ed_gt_sitk = sitk.ReadImage(str(ed_gt_path))
    es_gt_sitk = sitk.ReadImage(str(es_gt_path))
    ed_gt = np.transpose(sitk.GetArrayFromImage(ed_gt_sitk))  # (x, y, z)
    es_gt = np.transpose(sitk.GetArrayFromImage(es_gt_sitk))  # (x, y, z)
    
    # Prepare transforms (images are already in [0, 1] range from preprocessing)
    transform = Compose([
        SpatialPadd(keys=view, spatial_size=(192, 192, 16), method="end"),
    ])
    
    n_slices, n_frames = images.shape[-2:]
    
    seg_task = None
    if debug_tracker is not None:
        seg_task = debug_tracker.start_task(f"SEGMENT ({patient_id} - {model_name})", gpu_task=torch.cuda.is_available())
    
    try:
        # Run inference on all frames
        labels_list = []
        for t in range(n_frames):
            # Convert to float and add batch dimension
            batch = {view: torch.from_numpy(images[None, ..., t].astype(np.float32) / 255.0)}
            batch = transform(batch)
            batch = {k: v[None, ...].to(device=device, dtype=torch.float32) for k, v in batch.items()}
            
            with torch.no_grad(), torch.autocast("cuda", dtype=dtype, enabled=torch.cuda.is_available()):
                logits = model(batch)[view]  # (1, 4, x, y, z)
            labels_list.append(torch.argmax(logits, dim=1)[0, ..., :n_slices])
        
        labels = torch.stack(labels_list, dim=-1).detach().to(torch.float32).cpu().numpy()  # (x, y, z, t)
        if seg_task is not None:
            debug_tracker.end_task(seg_task, success=True)
    except Exception:
        if seg_task is not None:
            debug_tracker.end_task(seg_task, success=False)
        raise
    
    # Get ED and ES frame predictions
    lv_volumes = [np.sum(labels[..., t] == 3) for t in range(n_frames)]
    ed_pred_idx = 0  # First frame is typically ED
    es_pred_idx = np.argmin(lv_volumes)  # Frame with minimum LV volume is ES
    
    ed_pred = labels[..., ed_pred_idx]
    es_pred = labels[..., es_pred_idx]
    
    metrics_task = None
    if debug_tracker is not None:
        metrics_task = debug_tracker.start_task(f"COMPUTE_METRICS ({patient_id} - {model_name})", gpu_task=False)
    
    # Get voxel spacing for HD95 calculation (from ground truth)
    spacing = ed_gt_sitk.GetSpacing()  # (x, y, z) in mm
    voxelspacing = spacing[::-1]  # Convert to (z, y, x) for numpy array order
    
    # Calculate Dice scores and HD95 for each class (RV=1, MYO=2, LV=3)
    results = {
        'patient_id': patient_id,
        'model': model_name,
        'ed_dice_rv': dice_score(ed_pred, ed_gt, 1),
        'ed_dice_myo': dice_score(ed_pred, ed_gt, 2),
        'ed_dice_lv': dice_score(ed_pred, ed_gt, 3),
        'es_dice_rv': dice_score(es_pred, es_gt, 1),
        'es_dice_myo': dice_score(es_pred, es_gt, 2),
        'es_dice_lv': dice_score(es_pred, es_gt, 3),
        'ed_pred_frame': ed_pred_idx,
        'es_pred_frame': es_pred_idx,
        # HD95 metrics
        'ed_hd95_rv': hd95_score(ed_pred, ed_gt, 1, voxelspacing),
        'ed_hd95_myo': hd95_score(ed_pred, ed_gt, 2, voxelspacing),
        'ed_hd95_lv': hd95_score(ed_pred, ed_gt, 3, voxelspacing),
        'es_hd95_rv': hd95_score(es_pred, es_gt, 1, voxelspacing),
        'es_hd95_myo': hd95_score(es_pred, es_gt, 2, voxelspacing),
        'es_hd95_lv': hd95_score(es_pred, es_gt, 3, voxelspacing),
    }
    
    # Calculate mean Dice
    results['ed_dice_mean'] = np.mean([results['ed_dice_rv'], results['ed_dice_myo'], results['ed_dice_lv']])
    results['es_dice_mean'] = np.mean([results['es_dice_rv'], results['es_dice_myo'], results['es_dice_lv']])
    results['overall_dice_mean'] = np.mean([results['ed_dice_mean'], results['es_dice_mean']])
    
    # Calculate mean HD95 (only for finite values)
    ed_hd95_vals = [results['ed_hd95_rv'], results['ed_hd95_myo'], results['ed_hd95_lv']]
    es_hd95_vals = [results['es_hd95_rv'], results['es_hd95_myo'], results['es_hd95_lv']]
    results['ed_hd95_mean'] = np.mean([v for v in ed_hd95_vals if np.isfinite(v)]) if any(np.isfinite(v) for v in ed_hd95_vals) else np.inf
    results['es_hd95_mean'] = np.mean([v for v in es_hd95_vals if np.isfinite(v)]) if any(np.isfinite(v) for v in es_hd95_vals) else np.inf
    results['overall_hd95_mean'] = np.mean([results['ed_hd95_mean'], results['es_hd95_mean']]) if np.isfinite(results['ed_hd95_mean']) and np.isfinite(results['es_hd95_mean']) else np.inf
    
    # Save ED prediction
    ed_pred_sitk = sitk.GetImageFromArray(np.transpose(ed_pred).astype(np.uint8))
    ed_pred_sitk.SetSpacing(ed_gt_sitk.GetSpacing())
    ed_pred_sitk.SetOrigin(ed_gt_sitk.GetOrigin())
    ed_pred_sitk.SetDirection(ed_gt_sitk.GetDirection())
    ed_filename = f"{patient_id}_ED_{model_name}.nii.gz"
    sitk.WriteImage(ed_pred_sitk, str(predictions_dir / ed_filename), useCompression=True)
    
    # Save ES prediction
    es_pred_sitk = sitk.GetImageFromArray(np.transpose(es_pred).astype(np.uint8))
    es_pred_sitk.SetSpacing(es_gt_sitk.GetSpacing())
    es_pred_sitk.SetOrigin(es_gt_sitk.GetOrigin())
    es_pred_sitk.SetDirection(es_gt_sitk.GetDirection())
    es_filename = f"{patient_id}_ES_{model_name}.nii.gz"
    sitk.WriteImage(es_pred_sitk, str(predictions_dir / es_filename), useCompression=True)
    
    if metrics_task is not None:
        debug_tracker.end_task(metrics_task, success=True)
    
    return results

In [ ]:
# Set device and dtype
dtype, device = torch.float32, torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
    if torch.cuda.is_bf16_supported():
        dtype = torch.bfloat16

print(f"Using device: {device}")
print(f"Using dtype: {dtype}")

## Configuration

Set paths for ACDC data, preprocessed outputs, and model files.

In [ ]:
# Configuration
CINEMA_ROOT = Path(os.environ["CINEMA_ROOT"])
ACDC_DATA_DIR = Path(os.environ["ACDC_ROOT"] + "/testing")
PREPROCESSED_DIR = CINEMA_ROOT / "preprocessed"
LOCAL_MODEL_DIR = Path(CINEMA_ROOT / "models")

# Results folder structure (organized by model)
RESULTS_DIR = CINEMA_ROOT / "results"
TEMP_INPUT_DIR = RESULTS_DIR / "temp_input"

# CASC-style debug output directories
DEBUG_DIR = RESULTS_DIR / "debug"
DEBUG_TEXT_DIR = DEBUG_DIR / "text"
DEBUG_FIGURES_DIR = DEBUG_DIR / "figures"
DEBUG_RUN_NAME = "notebook_cinema_acdc_testing"

# Create base output directories
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TEMP_INPUT_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_TEXT_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Auto-discover available models and seeds from the model directory
SEGMENTATION_DIR = LOCAL_MODEL_DIR / "finetuned" / "segmentation"
available_models = {}

if SEGMENTATION_DIR.exists():
    for model_dir in sorted(SEGMENTATION_DIR.iterdir()):
        if model_dir.is_dir() and model_dir.name.endswith("_sax"):
            model_name = model_dir.name.replace("_sax", "")
            # Find all seed files (e.g., acdc_sax_0.safetensors, acdc_sax_1.safetensors)
            seed_files = sorted(model_dir.glob(f"{model_dir.name}_*.safetensors"))
            seeds = []
            for sf in seed_files:
                # Extract seed number from filename
                try:
                    seed_num = int(sf.stem.split("_")[-1])
                    seeds.append(seed_num)
                except ValueError:
                    continue
            if seeds:
                available_models[model_name] = sorted(seeds)

print(f"CineMA root: {CINEMA_ROOT}")
print(f"ACDC testing data: {ACDC_DATA_DIR}")
print(f"Preprocessed data: {PREPROCESSED_DIR}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Debug directory: {DEBUG_DIR}")
print(f"Models loaded from: {LOCAL_MODEL_DIR}")

print(f"\n{'='*60}")
print("DISCOVERED MODELS AND SEEDS")
print(f"{'='*60}")
for model_name, seeds in available_models.items():
    print(f"  {model_name}: seeds {seeds}")
print(f"{'='*60}")

# Create per-model directory structure
for model_name in available_models.keys():
    model_results_dir = RESULTS_DIR / model_name
    (model_results_dir / "predictions").mkdir(parents=True, exist_ok=True)
    (model_results_dir / "visualizations").mkdir(parents=True, exist_ok=True)
    for seed in available_models[model_name]:
        seed_dir = model_results_dir / f"seed_{seed}"
        (seed_dir / "predictions").mkdir(parents=True, exist_ok=True)
        (seed_dir / "visualizations").mkdir(parents=True, exist_ok=True)

# Also create combined results directory
(RESULTS_DIR / "combined").mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "combined" / "predictions").mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "combined" / "visualizations").mkdir(parents=True, exist_ok=True)

print(f"\nCreated directory structure:")
print(f"  {RESULTS_DIR}/")
print(f"    debug/              - CASC-style debug reports")
print(f"    combined/           - All models combined")
for model_name in available_models.keys():
    print(f"    {model_name}/")
    print(f"      predictions/      - Averaged across seeds")
    print(f"      visualizations/")
    for seed in available_models[model_name]:
        print(f"      seed_{seed}/       - Individual seed results")

## Step 1: Preprocess ACDC Testing Data

Preprocess all 50 testing patients following the ACDC preprocessing pipeline:
- Resample to 1.0 x 1.0 x 10.0 mm spacing
- Crop to 192 x 192 based on LV center at ED frame
- Normalize intensity to [0, 1]

In [ ]:
# Step 1: Preprocess all patients
print("Preprocessing ACDC testing dataset...")
print(f"Input: {ACDC_DATA_DIR}")
print(f"Output: {PREPROCESSED_DIR}")

# Ensure output directory exists
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Process each patient individually using the official preprocessing
patient_dirs = sorted([d for d in ACDC_DATA_DIR.iterdir() if d.is_dir()])
print(f"Found {len(patient_dirs)} patients to process\n")

# Create a metadata dataframe to store results
metadata_list = []

for patient_dir in patient_dirs:
    patient_id = patient_dir.name
    config_path = patient_dir / "Info.cfg"
    
    if not config_path.exists():
        print(f"Warning: {patient_id} missing Info.cfg, skipping")
        continue
    
    print(f"Processing {patient_id}...")
    debug_task = DEBUG_TRACKER.start_task(f"PREPROCESS ({patient_id})", gpu_task=False)
    
    # Use the official preprocessing function
    try:
        metadata = preprocess_pid(config_path, PREPROCESSED_DIR)
        metadata_list.append(metadata)
        DEBUG_TRACKER.end_task(debug_task, success=True)
        print(f"  ✓ Saved to {PREPROCESSED_DIR / patient_id}")
    except Exception as e:
        DEBUG_TRACKER.end_task(debug_task, success=False)
        print(f"  ✗ Error processing {patient_id}: {e}")

# Save metadata to CSV
import pandas as pd
metadata_df = pd.DataFrame(metadata_list)
metadata_csv = PREPROCESSED_DIR / "metadata.csv"
metadata_df.to_csv(metadata_csv, index=False)
print(f"\n✓ Preprocessing complete!")
print(f"✓ Metadata saved to {metadata_csv}")

## Step 2: Run Inference on All Patients

Run segmentation inference using all 3 models (ACDC, M&Ms, M&Ms-2) with 3 seeds each.

In [ ]:
# Get all preprocessed patients
patient_dirs = sorted(PREPROCESSED_DIR.glob("patient*"))
patient_ids = [p.name for p in patient_dirs]
print(f"Found {len(patient_ids)} preprocessed patients")

# Run inference for all discovered models and seeds
all_results = []

for trained_dataset, seeds in available_models.items():
    for seed in seeds:
        print(f"\n{'='*60}")
        print(f"Running inference with {trained_dataset.upper()} model (seed {seed})")
        print(f"{'='*60}\n")
        
        # Get per-seed predictions directory
        seed_predictions_dir = RESULTS_DIR / trained_dataset / f"seed_{seed}" / "predictions"
        
        for patient_id in tqdm(patient_ids, desc=f"{trained_dataset}_seed{seed}"):
            try:
                results = run_acdc_inference(
                    preprocessed_dir=PREPROCESSED_DIR,
                    patient_id=patient_id,
                    trained_dataset=trained_dataset,
                    seed=seed,
                    device=device,
                    dtype=dtype,
                    local_model_dir=LOCAL_MODEL_DIR,
                    predictions_dir=seed_predictions_dir,
                    debug_tracker=DEBUG_TRACKER,
                )
                # Add model and seed info explicitly
                results['model_name'] = trained_dataset
                results['seed'] = seed
                all_results.append(results)
            except Exception as e:
                print(f"\n  ERROR processing {patient_id} with {trained_dataset}_seed{seed}: {e}")
                continue

# Save all results to CSV (combined)
results_df = pd.DataFrame(all_results)
results_csv_path = RESULTS_DIR / "combined" / "metrics_detailed.csv"
results_df.to_csv(results_csv_path, index=False)

# Also save per-model detailed results
for model_name in available_models.keys():
    model_data = results_df[results_df['model_name'] == model_name]
    model_csv_path = RESULTS_DIR / model_name / "metrics_detailed.csv"
    model_data.to_csv(model_csv_path, index=False)
    
    # Per-seed results
    for seed in available_models[model_name]:
        seed_results = model_data[model_data['seed'] == seed]
        seed_csv_path = RESULTS_DIR / model_name / f"seed_{seed}" / "metrics_detailed.csv"
        seed_results.to_csv(seed_csv_path, index=False)

print(f"\n✓ Inference complete!")
print(f"✓ Combined metrics saved to: {results_csv_path}")
for model_name in available_models.keys():
    print(f"✓ {model_name} metrics saved to: {RESULTS_DIR / model_name / 'metrics_detailed.csv'}")

## Step 3: Analyze Results

Calculate summary statistics and display Dice scores for each model.

In [ ]:
# Load results
results_df = pd.read_csv(RESULTS_DIR / "combined" / "metrics_detailed.csv")

# Helper function for safe statistics
def safe_stats(series):
    finite_vals = series[np.isfinite(series)]
    if len(finite_vals) > 0:
        return finite_vals.mean(), finite_vals.std()
    return np.nan, np.nan

def create_patient_summary(df, model_name=None):
    """Create patient-level summary from results dataframe."""
    patient_results = []
    
    for patient_id in sorted(df['patient_id'].unique()):
        patient_data = df[df['patient_id'] == patient_id]
        
        patient_result = {
            'patient_id': patient_id,
            'ed_dice_rv': patient_data['ed_dice_rv'].mean(),
            'ed_dice_myo': patient_data['ed_dice_myo'].mean(),
            'ed_dice_lv': patient_data['ed_dice_lv'].mean(),
            'ed_dice_mean': patient_data['ed_dice_mean'].mean(),
            'es_dice_rv': patient_data['es_dice_rv'].mean(),
            'es_dice_myo': patient_data['es_dice_myo'].mean(),
            'es_dice_lv': patient_data['es_dice_lv'].mean(),
            'es_dice_mean': patient_data['es_dice_mean'].mean(),
            'ed_hd95_rv': patient_data['ed_hd95_rv'].replace([np.inf, -np.inf], np.nan).mean(),
            'ed_hd95_myo': patient_data['ed_hd95_myo'].replace([np.inf, -np.inf], np.nan).mean(),
            'ed_hd95_lv': patient_data['ed_hd95_lv'].replace([np.inf, -np.inf], np.nan).mean(),
            'ed_hd95_mean': patient_data['ed_hd95_mean'].replace([np.inf, -np.inf], np.nan).mean(),
            'es_hd95_rv': patient_data['es_hd95_rv'].replace([np.inf, -np.inf], np.nan).mean(),
            'es_hd95_myo': patient_data['es_hd95_myo'].replace([np.inf, -np.inf], np.nan).mean(),
            'es_hd95_lv': patient_data['es_hd95_lv'].replace([np.inf, -np.inf], np.nan).mean(),
            'es_hd95_mean': patient_data['es_hd95_mean'].replace([np.inf, -np.inf], np.nan).mean(),
            'overall_dice_mean': patient_data['overall_dice_mean'].mean(),
            'overall_hd95_mean': patient_data['overall_hd95_mean'].replace([np.inf, -np.inf], np.nan).mean(),
        }
        if model_name:
            patient_result['model_name'] = model_name
        patient_results.append(patient_result)
    
    return pd.DataFrame(patient_results)

def print_summary_stats(patient_df, title):
    """Print summary statistics for a patient dataframe."""
    print(f"\n{'='*80}")
    print(f"{title} - DICE SCORES")
    print(f"{'='*80}")
    
    print("\n--- End-Diastole (ED) Dice Scores ---")
    print(f"  RV: {patient_df['ed_dice_rv'].mean():.4f} ± {patient_df['ed_dice_rv'].std():.4f}")
    print(f"  MYO: {patient_df['ed_dice_myo'].mean():.4f} ± {patient_df['ed_dice_myo'].std():.4f}")
    print(f"  LV: {patient_df['ed_dice_lv'].mean():.4f} ± {patient_df['ed_dice_lv'].std():.4f}")
    print(f"  MEAN: {patient_df['ed_dice_mean'].mean():.4f} ± {patient_df['ed_dice_mean'].std():.4f}")
    
    print("\n--- End-Systole (ES) Dice Scores ---")
    print(f"  RV: {patient_df['es_dice_rv'].mean():.4f} ± {patient_df['es_dice_rv'].std():.4f}")
    print(f"  MYO: {patient_df['es_dice_myo'].mean():.4f} ± {patient_df['es_dice_myo'].std():.4f}")
    print(f"  LV: {patient_df['es_dice_lv'].mean():.4f} ± {patient_df['es_dice_lv'].std():.4f}")
    print(f"  MEAN: {patient_df['es_dice_mean'].mean():.4f} ± {patient_df['es_dice_mean'].std():.4f}")
    
    print("\n--- Overall ---")
    print(f"  Dice MEAN: {patient_df['overall_dice_mean'].mean():.4f} ± {patient_df['overall_dice_mean'].std():.4f}")
    mean_hd95, std_hd95 = safe_stats(patient_df['overall_hd95_mean'])
    print(f"  HD95 MEAN: {mean_hd95:.2f} ± {std_hd95:.2f} mm")

# Get unique models from results
unique_models = results_df['model_name'].unique() if 'model_name' in results_df.columns else []

# Create patient summaries per model (averaged across seeds)
model_patient_dfs = {}
for model_name in unique_models:
    model_data = results_df[results_df['model_name'] == model_name]
    model_patient_df = create_patient_summary(model_data, model_name)
    model_patient_dfs[model_name] = model_patient_df
    
    # Save per-model patient summary
    model_csv_path = RESULTS_DIR / model_name / "metrics_by_patient.csv"
    model_patient_df.to_csv(model_csv_path, index=False)
    print(f"✓ {model_name} patient-level results saved to: {model_csv_path}")
    
    # Print statistics
    print_summary_stats(model_patient_df, f"CineMA {model_name.upper()} MODEL")

# Also create combined patient summary (all models)
patient_df = create_patient_summary(results_df)
patient_csv_path = RESULTS_DIR / "combined" / "metrics_by_patient.csv"
patient_df.to_csv(patient_csv_path, index=False)
print(f"\n✓ Combined patient-level results saved to: {patient_csv_path}")
print_summary_stats(patient_df, "CineMA ALL MODELS COMBINED")

# Create summary table for combined results
summary_data = {
    'Metric': ['Dice RV', 'Dice MYO', 'Dice LV', 'Dice Mean', 'HD95 RV', 'HD95 MYO', 'HD95 LV', 'HD95 Mean'],
    'ED Mean': [
        patient_df['ed_dice_rv'].mean(),
        patient_df['ed_dice_myo'].mean(),
        patient_df['ed_dice_lv'].mean(),
        patient_df['ed_dice_mean'].mean(),
        safe_stats(patient_df['ed_hd95_rv'])[0],
        safe_stats(patient_df['ed_hd95_myo'])[0],
        safe_stats(patient_df['ed_hd95_lv'])[0],
        safe_stats(patient_df['ed_hd95_mean'])[0],
    ],
    'ED Std': [
        patient_df['ed_dice_rv'].std(),
        patient_df['ed_dice_myo'].std(),
        patient_df['ed_dice_lv'].std(),
        patient_df['ed_dice_mean'].std(),
        safe_stats(patient_df['ed_hd95_rv'])[1],
        safe_stats(patient_df['ed_hd95_myo'])[1],
        safe_stats(patient_df['ed_hd95_lv'])[1],
        safe_stats(patient_df['ed_hd95_mean'])[1],
    ],
    'ES Mean': [
        patient_df['es_dice_rv'].mean(),
        patient_df['es_dice_myo'].mean(),
        patient_df['es_dice_lv'].mean(),
        patient_df['es_dice_mean'].mean(),
        safe_stats(patient_df['es_hd95_rv'])[0],
        safe_stats(patient_df['es_hd95_myo'])[0],
        safe_stats(patient_df['es_hd95_lv'])[0],
        safe_stats(patient_df['es_hd95_mean'])[0],
    ],
    'ES Std': [
        patient_df['es_dice_rv'].std(),
        patient_df['es_dice_myo'].std(),
        patient_df['es_dice_lv'].std(),
        patient_df['es_dice_mean'].std(),
        safe_stats(patient_df['es_hd95_rv'])[1],
        safe_stats(patient_df['es_hd95_myo'])[1],
        safe_stats(patient_df['es_hd95_lv'])[1],
        safe_stats(patient_df['es_hd95_mean'])[1],
    ],
}

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*80)
print("COMBINED SUMMARY TABLE")
print("="*80)
print(summary_df.to_string(index=False))

# Save summary statistics
summary_csv_path = RESULTS_DIR / "combined" / "metrics_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)
print(f"\n✓ Combined summary statistics saved to: {summary_csv_path}")

In [ ]:
"""Create box plots for Dice scores and HD95 - per model and combined."""

def create_boxplot(patient_df, title, save_path):
    """Create standardized boxplot for a given patient dataframe."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Dice scores - ED
    dice_ed_data = [patient_df['ed_dice_rv'], patient_df['ed_dice_myo'], patient_df['ed_dice_lv']]
    bp1 = axes[0, 0].boxplot(dice_ed_data, labels=['RV', 'MYO', 'LV'], patch_artist=True)
    colors = ['red', 'green', 'blue']
    for patch, color in zip(bp1['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.5)
    axes[0, 0].set_title('ED Dice Scores', fontsize=12)
    axes[0, 0].set_ylabel('Dice Score')
    axes[0, 0].set_ylim([0, 1])
    axes[0, 0].axhline(y=0.9, color='gray', linestyle='--', alpha=0.5)
    
    # Dice scores - ES
    dice_es_data = [patient_df['es_dice_rv'], patient_df['es_dice_myo'], patient_df['es_dice_lv']]
    bp2 = axes[0, 1].boxplot(dice_es_data, labels=['RV', 'MYO', 'LV'], patch_artist=True)
    for patch, color in zip(bp2['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.5)
    axes[0, 1].set_title('ES Dice Scores', fontsize=12)
    axes[0, 1].set_ylabel('Dice Score')
    axes[0, 1].set_ylim([0, 1])
    axes[0, 1].axhline(y=0.9, color='gray', linestyle='--', alpha=0.5)
    
    # HD95 - ED (filter out infinite values)
    hd95_ed_data = [
        patient_df['ed_hd95_rv'][np.isfinite(patient_df['ed_hd95_rv'])],
        patient_df['ed_hd95_myo'][np.isfinite(patient_df['ed_hd95_myo'])],
        patient_df['ed_hd95_lv'][np.isfinite(patient_df['ed_hd95_lv'])]
    ]
    bp3 = axes[1, 0].boxplot(hd95_ed_data, labels=['RV', 'MYO', 'LV'], patch_artist=True)
    for patch, color in zip(bp3['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.5)
    axes[1, 0].set_title('ED HD95 (mm)', fontsize=12)
    axes[1, 0].set_ylabel('HD95 (mm)')
    
    # HD95 - ES
    hd95_es_data = [
        patient_df['es_hd95_rv'][np.isfinite(patient_df['es_hd95_rv'])],
        patient_df['es_hd95_myo'][np.isfinite(patient_df['es_hd95_myo'])],
        patient_df['es_hd95_lv'][np.isfinite(patient_df['es_hd95_lv'])]
    ]
    bp4 = axes[1, 1].boxplot(hd95_es_data, labels=['RV', 'MYO', 'LV'], patch_artist=True)
    for patch, color in zip(bp4['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.5)
    axes[1, 1].set_title('ES HD95 (mm)', fontsize=12)
    axes[1, 1].set_ylabel('HD95 (mm)')
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    # Save the figure
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"✓ Box plots saved to: {save_path}")
    
    plt.show()

# Create boxplots for each model (averaged across seeds)
print("Creating per-model boxplots (averaged across seeds)...")
print("="*80)

for model_name, model_patient_df in model_patient_dfs.items():
    title = f'CineMA {model_name.upper()} ACDC Segmentation Metrics Distribution'
    save_path = RESULTS_DIR / model_name / "metrics_boxplots.png"
    create_boxplot(model_patient_df, title, save_path)

# Create combined boxplot (all models)
print("\nCreating combined boxplot (all models)...")
print("="*80)
title = 'CineMA ACDC Segmentation Metrics Distribution (All Models)'
save_path = RESULTS_DIR / "combined" / "metrics_boxplots.png"
create_boxplot(patient_df, title, save_path)

## Output Files Summary

The following files are saved in the `results/` directory with organized subfolders:

**Directory Structure:**
```
results/
├── combined/                    # All models combined
│   ├── metrics_detailed.csv     # All predictions, all models
│   ├── metrics_by_patient.csv   # Patient-level (all models averaged)
│   ├── metrics_summary.csv      # Summary statistics
│   └── metrics_boxplots.png     # Combined boxplots
├── {model_name}/                # Per-model results (e.g., acdc/, mnms/, mnms2/)
│   ├── metrics_detailed.csv     # All seeds for this model
│   ├── metrics_by_patient.csv   # Patient-level (seeds averaged)
│   ├── metrics_boxplots.png     # Model boxplots (seeds averaged)
│   └── seed_{N}/                # Per-seed results
│       ├── predictions/         # NIfTI segmentation files
│       ├── visualizations/      # Comparison images
│       └── metrics_detailed.csv # Individual seed metrics
└── temp_input/                  # Temporary processing files
```

**Predictions** (`results/{model}/seed_{N}/predictions/`):
- `patient###_ED_{model}_seed{seed}.nii.gz` - ED frame predictions
- `patient###_ES_{model}_seed{seed}.nii.gz` - ES frame predictions

**Metrics**:
- Per-seed: `results/{model}/seed_{N}/metrics_detailed.csv`
- Per-model (averaged): `results/{model}/metrics_by_patient.csv`
- Combined: `results/combined/metrics_*.csv`

**Visualizations**:
- Per-model boxplots: `results/{model}/metrics_boxplots.png`
- Combined boxplots: `results/combined/metrics_boxplots.png`

## Step 4: Visualize Predictions vs Ground Truth

Compare predicted segmentations with ground truth for qualitative assessment.

In [ ]:
def load_nifti_image(filepath):
    """Load NIfTI image using SimpleITK."""
    image = sitk.ReadImage(str(filepath))
    array = sitk.GetArrayFromImage(image)
    return np.transpose(array)  # Convert from (z, y, x) to (x, y, z)


def visualize_comparison(pred_path, gt_path, original_path, patient_id, frame_type, model_name, save_path=None):
    """Visualize prediction vs ground truth comparison."""
    # Load images
    pred_array = load_nifti_image(pred_path)  # (x, y, z)
    gt_array = load_nifti_image(gt_path)  # (x, y, z)
    original_array = load_nifti_image(original_path)  # (x, y, z)
    
    # Select middle slice for visualization
    mid_slice = pred_array.shape[2] // 2
    
    pred_slice = pred_array[:, :, mid_slice]
    gt_slice = gt_array[:, :, mid_slice]
    original_slice = original_array[:, :, mid_slice]
    
    # Calculate Dice scores for this volume (full 3D)
    dsc_rv = dice_score(pred_array, gt_array, 1)
    dsc_myo = dice_score(pred_array, gt_array, 2)
    dsc_lv = dice_score(pred_array, gt_array, 3)
    dsc_mean = (dsc_rv + dsc_myo + dsc_lv) / 3
    
    # Create visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Colormap for segmentation (0=BG, 1=RV, 2=MYO, 3=LV)
    cmap = plt.cm.colors.ListedColormap(['black', 'red', 'green', 'blue'])
    
    # Top row: Original, Prediction, Ground Truth
    axes[0, 0].imshow(original_slice.T, cmap='gray', origin='lower')
    axes[0, 0].set_title(f'Original - {patient_id} ({frame_type}, Slice {mid_slice})', fontsize=12)
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(pred_slice.T, cmap=cmap, origin='lower', vmin=0, vmax=3)
    axes[0, 1].set_title(f'Prediction ({model_name})', fontsize=12)
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(gt_slice.T, cmap=cmap, origin='lower', vmin=0, vmax=3)
    axes[0, 2].set_title('Ground Truth', fontsize=12)
    axes[0, 2].axis('off')
    
    # Bottom row: Overlays and Difference
    axes[1, 0].imshow(original_slice.T, cmap='gray', origin='lower')
    pred_masked = np.ma.masked_where(pred_slice == 0, pred_slice)
    axes[1, 0].imshow(pred_masked.T, alpha=0.6, cmap=cmap, origin='lower', vmin=0, vmax=3)
    axes[1, 0].set_title('Prediction Overlay', fontsize=12)
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(original_slice.T, cmap='gray', origin='lower')
    gt_masked = np.ma.masked_where(gt_slice == 0, gt_slice)
    axes[1, 1].imshow(gt_masked.T, alpha=0.6, cmap=cmap, origin='lower', vmin=0, vmax=3)
    axes[1, 1].set_title('Ground Truth Overlay', fontsize=12)
    axes[1, 1].axis('off')
    
    difference = np.abs(pred_slice.astype(int) - gt_slice.astype(int))
    axes[1, 2].imshow(difference.T, cmap='hot', origin='lower')
    axes[1, 2].set_title(f'Difference Map\nMean DSC: {dsc_mean:.3f}', fontsize=12)
    axes[1, 2].axis('off')
    
    # Add legend
    legend_elements = [
        plt.Rectangle((0,0),1,1, facecolor='red', label=f'RV (DSC={dsc_rv:.3f})'),
        plt.Rectangle((0,0),1,1, facecolor='green', label=f'MYO (DSC={dsc_myo:.3f})'),
        plt.Rectangle((0,0),1,1, facecolor='blue', label=f'LV (DSC={dsc_lv:.3f})'),
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=11)
    
    plt.suptitle(f'{patient_id} - {frame_type} Frame - DSC: RV={dsc_rv:.3f} | MYO={dsc_myo:.3f} | LV={dsc_lv:.3f}', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  Saved: {save_path}")
    
    plt.show()


print("Visualization functions loaded successfully")

In [ ]:
# Visualize predictions for sample patients (using first available model and seed)
sample_patients = sorted(patient_df['patient_id'].unique())[:3]

# Get first available model and seed dynamically
first_model = list(available_models.keys())[0] if available_models else "acdc"
first_seed = available_models[first_model][0] if first_model in available_models else 0
best_model = f"{first_model}_seed{first_seed}"

print(f"Visualizing predictions for: {sample_patients}")
print(f"Using model: {best_model}\n")

# Get the predictions directory for this model/seed
predictions_dir = RESULTS_DIR / first_model / f"seed_{first_seed}" / "predictions"
visualizations_dir = RESULTS_DIR / first_model / f"seed_{first_seed}" / "visualizations"

for patient_id in sample_patients:
    print(f"\n{'='*80}")
    print(f"Patient: {patient_id}")
    print(f"{'='*80}")
    
    preprocessed_patient_dir = PREPROCESSED_DIR / patient_id
    
    # Visualize ED frame
    ed_pred_path = predictions_dir / f"{patient_id}_ED_{best_model}.nii.gz"
    ed_gt_path = preprocessed_patient_dir / f"{patient_id}_sax_ed_gt.nii.gz"
    ed_img_path = preprocessed_patient_dir / f"{patient_id}_sax_ed.nii.gz"
    
    if ed_pred_path.exists() and ed_gt_path.exists() and ed_img_path.exists():
        save_path = visualizations_dir / f"{patient_id}_ED_comparison.png"
        visualize_comparison(ed_pred_path, ed_gt_path, ed_img_path, 
                           patient_id, 'ED', best_model, save_path)
    else:
        print(f"Missing files for {patient_id} ED frame")
        print(f"  Checked: {ed_pred_path}")
    
    # Visualize ES frame
    es_pred_path = predictions_dir / f"{patient_id}_ES_{best_model}.nii.gz"
    es_gt_path = preprocessed_patient_dir / f"{patient_id}_sax_es_gt.nii.gz"
    es_img_path = preprocessed_patient_dir / f"{patient_id}_sax_es.nii.gz"
    
    if es_pred_path.exists() and es_gt_path.exists() and es_img_path.exists():
        save_path = visualizations_dir / f"{patient_id}_ES_comparison.png"
        visualize_comparison(es_pred_path, es_gt_path, es_img_path, 
                           patient_id, 'ES', best_model, save_path)
    else:
        print(f"Missing files for {patient_id} ES frame")
        print(f"  Checked: {es_pred_path}")

In [ ]:
"""Final summary."""

print("=" * 80)
print("CineMA ACDC SEGMENTATION - FINAL SUMMARY")
print("=" * 80)

print(f"\nDataset: ACDC Testing Set")
print(f"Number of patients: {len(patient_df)}")
print(f"Total predictions: {len(results_df)}")

# Count unique models and seeds
unique_models = results_df['model_name'].unique() if 'model_name' in results_df.columns else []
print(f"Models evaluated: {list(unique_models)}")
print(f"Seeds per model: {available_models}")

print("\n--- Per-Model Performance (averaged across seeds) ---")
for model_name, model_patient_df in model_patient_dfs.items():
    mean_dice = model_patient_df['overall_dice_mean'].mean()
    std_dice = model_patient_df['overall_dice_mean'].std()
    mean_hd95, std_hd95 = safe_stats(model_patient_df['overall_hd95_mean'])
    print(f"\n{model_name.upper()}:")
    print(f"  Mean Dice: {mean_dice:.4f} ± {std_dice:.4f}")
    print(f"  Mean HD95: {mean_hd95:.2f} ± {std_hd95:.2f} mm")

print("\n--- Combined Performance (all models) ---")
print(f"Mean Dice Score: {patient_df['overall_dice_mean'].mean():.4f} ± {patient_df['overall_dice_mean'].std():.4f}")
mean_hd95, std_hd95 = safe_stats(patient_df['overall_hd95_mean'])
print(f"Mean HD95: {mean_hd95:.2f} ± {std_hd95:.2f} mm")

print(f"\n--- Output Directory Structure ---")
print(f"Results directory: {RESULTS_DIR}")
print(f"  combined/")
print(f"    - metrics_detailed.csv")
print(f"    - metrics_by_patient.csv")
print(f"    - metrics_summary.csv")
print(f"    - metrics_boxplots.png")
for model_name in unique_models:
    model_dir = RESULTS_DIR / model_name
    pred_count = sum(1 for _ in model_dir.rglob("*.nii.gz"))
    vis_count = sum(1 for _ in model_dir.rglob("*.png"))
    print(f"  {model_name}/")
    print(f"    - metrics_by_patient.csv")
    print(f"    - metrics_boxplots.png")
    print(f"    - {len(available_models.get(model_name, []))} seed folders ({pred_count} predictions, {vis_count} visualizations)")

print("\n✓ Processing complete!")

In [ ]:
# Step 5: Export CASC-style debug artifacts
if 'results_df' not in globals() or results_df.empty:
    metrics_path = RESULTS_DIR / "combined" / "metrics_detailed.csv"
    if metrics_path.exists():
        results_df = pd.read_csv(metrics_path)
    else:
        raise RuntimeError("No metrics available. Run inference first.")

debug_report_task = DEBUG_TRACKER.start_task("GENERATE_REPORT", gpu_task=False)

# Build CASC-style metrics snapshot
snapshot_group_cols = ["patient_id", "model_name"] if "model_name" in results_df.columns else ["patient_id", "model"]
snapshot_metric_cols = [
    "ed_dice_rv", "ed_hd95_rv", "ed_dice_myo", "ed_hd95_myo", "ed_dice_lv", "ed_hd95_lv",
    "es_dice_rv", "es_hd95_rv", "es_dice_myo", "es_hd95_myo", "es_dice_lv", "es_hd95_lv",
    "overall_dice_mean",
]
snapshot_df = (
    results_df[snapshot_group_cols + snapshot_metric_cols]
    .groupby(snapshot_group_cols, as_index=False)
    .mean(numeric_only=True)
)
if "model_name" in snapshot_df.columns:
    snapshot_df["model"] = snapshot_df["model_name"].astype(str) + "__model"
    snapshot_df["architecture"] = snapshot_df["model_name"].astype(str)
else:
    snapshot_df["model"] = snapshot_df["model"].astype(str)
    snapshot_df["architecture"] = snapshot_df["model"].astype(str)

metrics_snapshot = snapshot_df[[
    "patient_id", "model", "architecture",
    "ed_dice_rv", "ed_hd95_rv", "ed_dice_myo", "ed_hd95_myo", "ed_dice_lv", "ed_hd95_lv",
    "es_dice_rv", "es_hd95_rv", "es_dice_myo", "es_hd95_myo", "es_dice_lv", "es_hd95_lv",
    "overall_dice_mean",
]].copy()
metrics_snapshot_path = DEBUG_TEXT_DIR / "metrics_snapshot.csv"
metrics_snapshot.to_csv(metrics_snapshot_path, index=False)

# Finalize debug report task and aggregate task profile outputs
DEBUG_TRACKER.end_task(debug_report_task, success=True)
task_df = pd.DataFrame(DEBUG_TRACKER.records)
if task_df.empty:
    task_profile = pd.DataFrame(columns=[
        "process", "tasks", "success_rate", "runtime_mean_s", "runtime_median_s", "runtime_p95_s",
        "cpu_percent_mean", "peak_rss_gb_mean", "peak_rss_gb_max", "runtime_hours_total",
    ])
else:
    task_profile = (
        task_df.groupby("process", as_index=False)
        .apply(
            lambda g: pd.Series(
                {
                    "tasks": int(len(g)),
                    "success_rate": _safe_float(g["success"].mean()),
                    "runtime_mean_s": _safe_float(g["runtime_s"].mean()),
                    "runtime_median_s": _safe_float(g["runtime_s"].median()),
                    "runtime_p95_s": _safe_float(np.percentile(g["runtime_s"], 95)),
                    "cpu_percent_mean": _safe_float(g["cpu_percent"].mean()),
                    "peak_rss_gb_mean": _safe_float(g["peak_rss_gb"].mean()),
                    "peak_rss_gb_max": _safe_float(g["peak_rss_gb"].max()),
                    "runtime_hours_total": _safe_float(g["runtime_s"].sum() / 3600.0),
                }
            )
        )
        .reset_index(drop=True)
        .sort_values("runtime_mean_s", ascending=False)
)

task_profile_path = DEBUG_TEXT_DIR / "task_profile.csv"
task_profile.to_csv(task_profile_path, index=False)

gpu_profile = task_df[task_df.get("gpu_task", False) == True].copy() if not task_df.empty else pd.DataFrame()
if gpu_profile.empty:
    gpu_profile_df = pd.DataFrame(columns=["process", "gpu_tasks", "gpu_walltime_hours", "runtime_median_s"])
else:
    gpu_profile_df = (
        gpu_profile.groupby("process", as_index=False)
        .apply(
            lambda g: pd.Series(
                {
                    "gpu_tasks": int(len(g)),
                    "gpu_walltime_hours": _safe_float(g["runtime_s"].sum() / 3600.0),
                    "runtime_median_s": _safe_float(g["runtime_s"].median()),
                }
            )
        )
        .reset_index(drop=True)
        .sort_values("runtime_median_s", ascending=False)
)

gpu_profile_path = DEBUG_TEXT_DIR / "gpu_profile.csv"
gpu_profile_df.to_csv(gpu_profile_path, index=False)

# Compute debug metrics summary using the same schema as CASC/VSA3L
workflow_duration_seconds = _safe_float(DEBUG_TRACKER.workflow_duration_seconds())
total_tasks = int(len(task_df))
completed_tasks = int(task_df["success"].sum()) if not task_df.empty else 0
failed_tasks = int(total_tasks - completed_tasks)
task_success_rate = _safe_float(completed_tasks / total_tasks) if total_tasks > 0 else float("nan")
retry_rate = 0.0

cpu_time_hours_est = float("nan")
if not task_df.empty:
    valid_cpu = task_df.dropna(subset=["cpu_percent", "runtime_s"]).copy()
    if not valid_cpu.empty:
        cpu_time_hours_est = _safe_float(((valid_cpu["cpu_percent"] / 100.0) * valid_cpu["runtime_s"]).sum() / 3600.0)

memory_gb_hours_est = float("nan")
if not task_df.empty:
    valid_mem = task_df.dropna(subset=["peak_rss_gb", "runtime_s"]).copy()
    if not valid_mem.empty:
        memory_gb_hours_est = _safe_float((valid_mem["peak_rss_gb"] * valid_mem["runtime_s"] / 3600.0).sum())

mean_parallelism_est = float("nan")
if workflow_duration_seconds > 0 and not task_df.empty:
    mean_parallelism_est = _safe_float(task_df["runtime_s"].sum() / workflow_duration_seconds)

n_patients_input = int(len(patient_ids)) if "patient_ids" in globals() else int(metrics_snapshot["patient_id"].nunique())
n_patients_with_metrics = int(metrics_snapshot["patient_id"].nunique()) if not metrics_snapshot.empty else 0
coverage_rate = _safe_float(n_patients_with_metrics / n_patients_input) if n_patients_input > 0 else float("nan")
patient_throughput_per_hour = _safe_float(n_patients_input * 3600.0 / workflow_duration_seconds) if workflow_duration_seconds > 0 else float("nan")

gpu_task_count = int(len(gpu_profile)) if not gpu_profile.empty else 0
gpu_task_fraction = _safe_float(gpu_task_count / total_tasks) if total_tasks > 0 else float("nan")
gpu_walltime_hours_est = _safe_float(gpu_profile["runtime_s"].sum() / 3600.0) if not gpu_profile.empty else 0.0
avg_gpu_concurrency_est = _safe_float((gpu_walltime_hours_est * 3600.0) / workflow_duration_seconds) if workflow_duration_seconds > 0 else float("nan")

n_models_evaluated = int(metrics_snapshot["model"].nunique()) if not metrics_snapshot.empty else 0
if not metrics_snapshot.empty:
    model_perf = metrics_snapshot.groupby("model", as_index=False)["overall_dice_mean"].mean()
    best_idx = model_perf["overall_dice_mean"].idxmax()
    best_model_overall_dice = str(model_perf.loc[best_idx, "model"] if pd.notna(best_idx) else "NA")
    best_model_dice_value = _safe_float(model_perf.loc[best_idx, "overall_dice_mean"] if pd.notna(best_idx) else float("nan"))
    cross_model_dice_spread = _safe_float(model_perf["overall_dice_mean"].std(ddof=0)) if len(model_perf) > 1 else float("nan")
else:
    best_model_overall_dice = "NA"
    best_model_dice_value = float("nan")
    cross_model_dice_spread = float("nan")

# Architecture consistency as mean CV of per-patient overall Dice within each architecture.
if not metrics_snapshot.empty and metrics_snapshot["architecture"].nunique() > 0:
    cv_values = []
    for _, g in metrics_snapshot.groupby("architecture"):
        mean_val = g["overall_dice_mean"].mean()
        std_val = g["overall_dice_mean"].std(ddof=0)
        if pd.notna(mean_val) and mean_val != 0 and pd.notna(std_val):
            cv_values.append(std_val / mean_val)
    architecture_consistency_cv_mean = _safe_float(np.mean(cv_values)) if cv_values else float("nan")
else:
    architecture_consistency_cv_mean = float("nan")

if not metrics_snapshot.empty:
    rv_dice_mean = _safe_float(((metrics_snapshot["ed_dice_rv"] + metrics_snapshot["es_dice_rv"]) / 2.0).mean())
    myo_dice_mean = _safe_float(((metrics_snapshot["ed_dice_myo"] + metrics_snapshot["es_dice_myo"]) / 2.0).mean())
    lv_dice_mean = _safe_float(((metrics_snapshot["ed_dice_lv"] + metrics_snapshot["es_dice_lv"]) / 2.0).mean())
else:
    rv_dice_mean = float("nan")
    myo_dice_mean = float("nan")
    lv_dice_mean = float("nan")

models_requested_str = ",".join(sorted([m.upper() for m in available_models.keys()])) if available_models else "NONE"

summary_payload = {
    "run_name": DEBUG_RUN_NAME,
    "workflow_start": DEBUG_TRACKER.workflow_start_iso,
    "models_requested": models_requested_str,
    "execution": {
        "workflow_success": failed_tasks == 0,
        "workflow_duration_seconds": workflow_duration_seconds,
        "total_tasks": total_tasks,
        "completed_tasks": completed_tasks,
        "failed_tasks": failed_tasks,
        "task_success_rate": task_success_rate,
        "retry_rate": retry_rate,
        "cpu_time_hours_est": cpu_time_hours_est,
        "memory_gb_hours_est": memory_gb_hours_est,
        "makespan_seconds_trace": workflow_duration_seconds,
        "mean_parallelism_est": mean_parallelism_est,
        "patient_throughput_per_hour": patient_throughput_per_hour,
    },
    "gpu": {
        "gpu_task_count": gpu_task_count,
        "gpu_task_fraction": gpu_task_fraction,
        "gpu_walltime_hours_est": gpu_walltime_hours_est,
        "avg_gpu_concurrency_est": avg_gpu_concurrency_est,
    },
    "scientific": {
        "n_patients_input": n_patients_input,
        "n_patients_with_metrics": n_patients_with_metrics,
        "coverage_rate": coverage_rate,
        "n_models_evaluated": n_models_evaluated,
        "best_model_overall_dice": best_model_overall_dice,
        "best_model_dice_value": best_model_dice_value,
        "cross_model_dice_spread": cross_model_dice_spread,
        "architecture_consistency_cv_mean": architecture_consistency_cv_mean,
        "rv_dice_mean": rv_dice_mean,
        "myo_dice_mean": myo_dice_mean,
        "lv_dice_mean": lv_dice_mean,
    },
}

summary_json_path = DEBUG_TEXT_DIR / "debug_metrics_summary.json"
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2, allow_nan=True)

# Markdown report in the same CASC style
report_lines = [
    "# CASC Debug Report",
    "",
    "## Run Context",
    f"- Run name: {summary_payload['run_name']}",
    f"- Models requested: {summary_payload['models_requested']}",
    f"- Workflow start: {summary_payload['workflow_start']}",
    f"- Workflow success: {summary_payload['execution']['workflow_success']}",
    "",
    "## Execution & Reliability",
    f"- Workflow duration (s): {_safe_float(summary_payload['execution']['workflow_duration_seconds']):.3f}",
    f"- Total tasks: {summary_payload['execution']['total_tasks']}",
    f"- Task success rate: {_safe_float(summary_payload['execution']['task_success_rate']):.4f}",
    f"- Retry rate: {_safe_float(summary_payload['execution']['retry_rate']):.4f}",
    f"- Estimated CPU time (core-hours): {_nan_to_na(summary_payload['execution']['cpu_time_hours_est'], precision=3)}",
    f"- Estimated memory pressure (GB-hours from peak RSS): {_nan_to_na(summary_payload['execution']['memory_gb_hours_est'], precision=3)}",
    f"- Mean parallelism estimate: {_nan_to_na(summary_payload['execution']['mean_parallelism_est'], precision=3)}",
    f"- Patient throughput (patients/hour): {_nan_to_na(summary_payload['execution']['patient_throughput_per_hour'], precision=3)}",
    "",
    "## GPU Scalability",
    f"- GPU task count: {summary_payload['gpu']['gpu_task_count']}",
    f"- GPU task fraction: {_nan_to_na(summary_payload['gpu']['gpu_task_fraction'], precision=4)}",
    f"- Estimated GPU walltime (hours): {_nan_to_na(summary_payload['gpu']['gpu_walltime_hours_est'], precision=3)}",
    f"- Average GPU concurrency estimate: {_nan_to_na(summary_payload['gpu']['avg_gpu_concurrency_est'], precision=3)}",
    "",
    "## Scientific Utility",
    f"- Input patient count: {summary_payload['scientific']['n_patients_input']}",
    f"- Evaluated patient count: {summary_payload['scientific']['n_patients_with_metrics']}",
    f"- Coverage rate: {_nan_to_na(summary_payload['scientific']['coverage_rate'], precision=4)}",
    f"- Models evaluated: {summary_payload['scientific']['n_models_evaluated']}",
    f"- Best model by mean overall Dice: {summary_payload['scientific']['best_model_overall_dice']} ({_nan_to_na(summary_payload['scientific']['best_model_dice_value'], precision=4)})",
    f"- Cross-model Dice spread: {_nan_to_na(summary_payload['scientific']['cross_model_dice_spread'], precision=4)}",
    f"- Architecture consistency CV (lower is better): {_nan_to_na(summary_payload['scientific']['architecture_consistency_cv_mean'], precision=4)}",
    f"- Mean RV Dice: {_nan_to_na(summary_payload['scientific']['rv_dice_mean'], precision=4)}",
    f"- Mean MYO Dice: {_nan_to_na(summary_payload['scientific']['myo_dice_mean'], precision=4)}",
    f"- Mean LV Dice: {_nan_to_na(summary_payload['scientific']['lv_dice_mean'], precision=4)}",
]

report_md_path = DEBUG_TEXT_DIR / "debug_report.md"
report_md_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

# Save simple debug figures to keep same debug folder form factor.
if not task_profile.empty:
    plt.figure(figsize=(14, 6))
    plot_df = task_profile.head(20).copy()
    plt.barh(plot_df["process"], plot_df["runtime_mean_s"], color="#4c78a8")
    plt.gca().invert_yaxis()
    plt.xlabel("Mean Runtime (s)")
    plt.title("Top 20 Processes by Mean Runtime")
    plt.tight_layout()
    plt.savefig(DEBUG_FIGURES_DIR / "process_runtime_top20.png", dpi=150)
    plt.close()

if not metrics_snapshot.empty:
    model_means = metrics_snapshot.groupby("model", as_index=False)["overall_dice_mean"].mean()
    plt.figure(figsize=(10, 5))
    plt.bar(model_means["model"], model_means["overall_dice_mean"], color="#f58518")
    plt.ylim(0, 1)
    plt.ylabel("Mean Overall Dice")
    plt.title("Mean Overall Dice by Model")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(DEBUG_FIGURES_DIR / "mean_overall_dice_by_model.png", dpi=150)
    plt.close()

print("✓ CASC-style debug artifacts created:")
print(f"  {summary_json_path}")
print(f"  {report_md_path}")
print(f"  {task_profile_path}")
print(f"  {gpu_profile_path}")
print(f"  {metrics_snapshot_path}")
print(f"  {DEBUG_FIGURES_DIR}")